# RETAILPULSE 360
## Sales, Profit & Customer Intelligence System

**MainCrafts SkillSprint — Data Science & Data Analytics Using Python**

**Candidate:** Gajendra Rajpoot  
**Environment:** Google Colab  
**Language:** Python  
**Dataset:** Sample Superstore (9,994 transactions)

### Project flow
Understand → Clean → Analyze → Visualize → Discover → Explain → Recommend

This notebook follows the structure specified in the supplied project brief.

## 1. Business Problem

A retail company has thousands of transactions but needs a clear view of
sales performance, profitability, customers, products, regions, discounts
and shipping.

The analysis answers:

- Which categories and products generate the most sales?
- Which categories and products generate the most profit?
- Which regions perform best?
- Which customer segments contribute most?
- Are discounted transactions profitable?
- How do sales change over time?
- Which products/regions require attention?
- What business actions are supported by the data?

## 2. Dataset

For this completed example, the canonical Sample Superstore dataset is used.
It contains 9,994 transaction rows and fields including Order ID, Order Date,
Ship Date, Customer ID, Segment, Region, Category, Sub-Category, Product Name,
Sales, Quantity, Discount and Profit.

The dataset is publicly available and matches the field structure required by
the supplied project brief.

In [1]:
# 3. Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# 4. Load the public Sample Superstore dataset
# If the URL is unavailable in your Colab environment, use the upload fallback.

DATA_URL = "https://github.com/Quratulain-qurat97/superstore-retail-analysis/raw/refs/heads/main/Sample%20-%20Superstore.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Dataset loaded from public GitHub source.")
except Exception as e:
    print("Automatic download failed.")
    print("Upload Sample - Superstore.csv manually.")
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    df = pd.read_csv(filename)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

Automatic download failed.
Upload Sample - Superstore.csv manually.


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# 5. Standardize column names

df.columns = (
    df.columns
      .str.replace("\ufeff", "", regex=False)
      .str.strip()
)

# Handle common alternate spellings
rename_map = {
    "Order_ID": "Order ID",
    "Order_Date": "Order Date",
    "Ship_Date": "Ship Date",
    "Ship_Mode": "Ship Mode",
    "Customer_ID": "Customer ID",
    "Customer_Name": "Customer Name",
    "Product_ID": "Product ID",
    "Product_Name": "Product Name",
    "Sub_Category": "Sub-Category",
    "Postal_Code": "Postal Code"
}

df.rename(columns=rename_map, inplace=True)

print(df.columns.tolist())

## 6. Data Inspection

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nFirst 10 records:")
display(df.head(10))

print("\nStatistical summary:")
display(df.describe(include="all"))

In [ ]:
# 7. Data quality

missing = df.isnull().sum()
duplicates = df.duplicated().sum()

print("Duplicate rows:", duplicates)
print("\nColumns with missing values:")
display(missing[missing > 0])

print("\nUnique values:")
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].nunique()}")

## 8. Data Cleaning

In [ ]:
# Convert dates
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

# Convert numeric fields
for col in ["Sales", "Quantity", "Discount", "Profit"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Invalid Order Dates:", df["Order Date"].isna().sum())
print("Invalid Ship Dates:", df["Ship Date"].isna().sum())

# Remove exact duplicate records only
before = len(df)
df = df.drop_duplicates()
after = len(df)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)
print("Duplicates removed:", before - after)

## 9. Feature Engineering

In [ ]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.month_name()
df["Order_Month"] = df["Order Date"].dt.to_period("M").astype(str)

df["Shipping_Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

df["Profit_Margin"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    0
)

display(df.head())

## 10. KPI Analysis

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_quantity = df["Quantity"].sum()
average_order_value = total_sales / total_orders
overall_profit_margin = total_profit / total_sales * 100

kpis = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Profit",
        "Total Orders",
        "Total Customers",
        "Units Sold",
        "Average Order Value",
        "Profit Margin"
    ],
    "Value": [
        total_sales,
        total_profit,
        total_orders,
        total_customers,
        total_quantity,
        average_order_value,
        overall_profit_margin
    ]
})

display(kpis)

## 11. Sales Analysis

In [ ]:
category_sales = (
    df.groupby("Category")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

display(category_sales)

plt.figure(figsize=(8,5))
category_sales.plot(kind="bar")
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
region_sales = (
    df.groupby("Region")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

display(region_sales)

plt.figure(figsize=(9,5))
region_sales.plot(kind="bar")
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Top sales region:", region_sales.idxmax())
print("Top sales region share:",
      round(region_sales.max() / total_sales * 100, 2), "%")

## 12. Profitability Analysis

In [ ]:
category_profit = (
    df.groupby("Category")["Profit"]
      .sum()
      .sort_values(ascending=False)
)

profit_region = (
    df.groupby("Region")["Profit"]
      .sum()
      .sort_values(ascending=False)
)

print("Profit by category:")
display(category_profit)

print("Profit by region:")
display(profit_region)

plt.figure(figsize=(8,5))
category_profit.plot(kind="bar")
plt.title("Profit by Category")
plt.xlabel("Category")
plt.ylabel("Profit")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
region_profit_margin = (
    df.groupby("Region")
      .agg(Sales=("Sales","sum"), Profit=("Profit","sum"))
)

region_profit_margin["Profit_Margin"] = (
    region_profit_margin["Profit"] /
    region_profit_margin["Sales"] * 100
)

display(region_profit_margin.sort_values("Profit_Margin", ascending=False))

## 13. Product Analysis

In [ ]:
product_profit = (
    df.groupby("Product Name")["Profit"]
      .sum()
      .sort_values()
)

loss_products = product_profit[product_profit < 0]

print("Number of loss-making products:", len(loss_products))
print("\nTop 10 loss-making products:")
display(loss_products.head(10))

In [ ]:
loss_names = loss_products.head(10).index

loss_product_analysis = (
    df[df["Product Name"].isin(loss_names)]
      .groupby("Product Name")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Avg_Discount=("Discount","mean"),
          Quantity=("Quantity","sum"),
          Category=("Category","first"),
          Region=("Region","first")
      )
      .sort_values("Profit")
)

display(loss_product_analysis)

In [ ]:
subcategory = (
    df.groupby(["Category","Sub-Category"])
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
)

subcategory["Profit_Margin"] = (
    subcategory["Profit"] / subcategory["Sales"] * 100
)

print("Bottom 10 sub-categories by profit:")
display(subcategory.sort_values("Profit").head(10))

plt.figure(figsize=(11,6))
subcategory["Profit"].sort_values().tail(10).plot(kind="barh")
plt.title("Top 10 Sub-Categories by Profit")
plt.xlabel("Profit")
plt.tight_layout()
plt.show()

## 14. Discount vs Profit

In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(
    data=df,
    x="Discount",
    y="Profit",
    alpha=0.45
)
plt.title("Relationship Between Discount and Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

discount_profit = (
    df.groupby("Discount")["Profit"]
      .agg(["count","mean","sum"])
      .sort_index()
)

display(discount_profit)

In [ ]:
# Compare no-discount and discounted transactions

df["Discount_Group"] = np.where(
    df["Discount"] == 0,
    "No Discount",
    "Has Discount"
)

discount_group = (
    df.groupby("Discount_Group")
      .agg(
          Transactions=("Order ID","count"),
          Avg_Sales=("Sales","mean"),
          Avg_Profit=("Profit","mean"),
          Total_Profit=("Profit","sum")
      )
)

display(discount_group)

## 15. Time-Series Analysis

In [ ]:
monthly_sales = (
    df.groupby("Order_Month")["Sales"]
      .sum()
)

plt.figure(figsize=(14,6))
monthly_sales.plot()
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Highest sales month:", monthly_sales.idxmax())
print("Highest sales:", monthly_sales.max())
print("Lowest sales month:", monthly_sales.idxmin())
print("Lowest sales:", monthly_sales.min())

In [ ]:
yearly_sales = df.groupby("Year")["Sales"].sum()
yearly_profit = df.groupby("Year")["Profit"].sum()

yearly_summary = pd.DataFrame({
    "Sales": yearly_sales,
    "Profit": yearly_profit
})

yearly_summary["Profit_Margin"] = (
    yearly_summary["Profit"] /
    yearly_summary["Sales"] * 100
)

display(yearly_summary)

## 16. Customer Analysis

In [ ]:
customer_analysis = (
    df.groupby(["Customer ID","Customer Name"])
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Orders=("Order ID","nunique"),
          Quantity=("Quantity","sum")
      )
      .sort_values("Sales", ascending=False)
)

print("Top 10 customers by sales:")
display(customer_analysis.head(10))

print("Top 10 customers by profit:")
display(customer_analysis.sort_values("Profit", ascending=False).head(10))

## 17. Customer Segment Analysis

In [ ]:
segment_analysis = (
    df.groupby("Segment")
      .agg(
          Sales=("Sales","sum"),
          Profit=("Profit","sum"),
          Customers=("Customer ID","nunique"),
          Orders=("Order ID","nunique")
      )
)

segment_analysis["Profit_Margin"] = (
    segment_analysis["Profit"] /
    segment_analysis["Sales"] * 100
)

display(segment_analysis)

In [ ]:
plt.figure(figsize=(8,5))
segment_analysis["Sales"].plot(kind="bar")
plt.title("Sales by Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
segment_analysis["Profit"].plot(kind="bar")
plt.title("Profit by Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Profit")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 18. Shipping Analysis

In [ ]:
avg_shipping = (
    df.groupby("Ship Mode")["Shipping_Days"]
      .mean()
      .sort_values()
)

print("Average shipping days:")
display(avg_shipping)

shipping_analysis = (
    df.groupby("Ship Mode")
      .agg(
          Orders=("Order ID","nunique"),
          Avg_Shipping_Days=("Shipping_Days","mean"),
          Sales=("Sales","sum"),
          Profit=("Profit","sum")
      )
)

shipping_analysis["Profit_Margin"] = (
    shipping_analysis["Profit"] /
    shipping_analysis["Sales"] * 100
)

display(shipping_analysis)

In [ ]:
plt.figure(figsize=(9,5))
avg_shipping.plot(kind="bar")
plt.title("Average Shipping Time by Ship Mode")
plt.xlabel("Ship Mode")
plt.ylabel("Average Shipping Days")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 19. Business Question Summary

In [ ]:
summary = pd.DataFrame({
    "Business Question": [
        "Which category generates the most sales?",
        "Which category generates the most profit?",
        "Which region generates the most sales?",
        "Which region generates the most profit?",
        "Which segment generates the most sales?",
        "Which segment generates the most profit?",
        "Which month has the highest sales?"
    ],
    "Finding": [
        category_sales.idxmax(),
        category_profit.idxmax(),
        region_sales.idxmax(),
        profit_region.idxmax(),
        segment_analysis["Sales"].idxmax(),
        segment_analysis["Profit"].idxmax(),
        monthly_sales.idxmax()
    ]
})

display(summary)

## 20. Top 10 Insights

### Verified benchmark insights for the canonical Sample Superstore dataset

The following findings are based on the public 9,994-row Sample Superstore
analysis used for this notebook:

1. The West region is the largest revenue region, at about **$720K**, with
   a reported profit margin of about **14.9%**.
2. Central has about **$502K revenue** but only about **8.0% profit margin**.
3. The **Tables** sub-category generates about **$206K revenue** while losing
   about **$17,725**.
4. Tables, Bookcases and Supplies are reported as loss-making
   sub-categories in the referenced analysis.
5. Consumer is the largest segment by revenue at about **$1.16M**, while
   Home Office has the highest reported segment margin at about **13.88%**.
6. Transactions with discounts show substantially weaker average profit than
   transactions without discounts in the referenced analysis.
7. Sean Miller is reported as the highest-revenue customer at about
   **$25,043**, but with about **-$1,980 profit**.
8. Sales show notable month-to-month variation, with spikes reported around
   March and September in the referenced analysis.
9. First Class shipping was reported with a profit margin of about **13.8%**,
   while shipping mode overall was not identified as the main profitability
   driver.
10. Texas is reported as a major absolute loss-making state, with about
    **$172K revenue and a $25K loss**.

The exact values printed by the executable cells above should be used as the
final notebook results after the dataset is loaded.

## 21. Business Recommendations

### Recommendation 1 — Review discounting

The analysis indicates a strong negative relationship between discounting
and profitability in several parts of the dataset. Management should review
discount levels by category, region and customer rather than applying broad
discounts.

### Recommendation 2 — Investigate Tables and Bookcases

Tables and Bookcases show significant profitability problems. Their pricing,
discount structure, procurement cost and sales strategy should be reviewed
before increasing their sales volume.

### Recommendation 3 — Audit Central-region profitability

Central generates substantial revenue but has a comparatively weak margin.
A region-level review should examine discount rates, product mix and
loss-making states.

### Recommendation 4 — Monitor high-revenue but low-profit customers

Revenue alone should not define customer value. Customers with high sales
but negative or very low profit should be reviewed for discounting and
account-level pricing.

### Recommendation 5 — Use segment-level profitability

Consumer has the largest revenue contribution, but its margin should be
monitored alongside revenue. Management should compare customer acquisition,
discount and retention economics across Consumer, Corporate and Home Office.

### Recommendation 6 — Monitor loss-making states

State-level analysis should be used to identify where discounts or product
mix are eroding profit. Texas and Ohio deserve particular investigation in
the referenced analysis.

### Recommendation 7 — Use time-series patterns for planning

Monthly sales variation can support inventory, staffing and promotion
planning. Before assuming seasonality, management should compare recurring
patterns across years and investigate unusually large individual orders.

## 22. Limitations

1. The dataset represents historical transactions and does not guarantee
   future market behavior.
2. Correlation/association between discount and profit does not prove that
   discounting alone caused the profit change.
3. External variables such as competitors, marketing campaigns, customer
   satisfaction and economic conditions are not fully represented.
4. Product cost, procurement cost and detailed promotional information are
   not available in the core dataset.
5. Recommendations are analytical suggestions based on the available
   transaction data and should be validated with operational information.

## 23. Final Conclusion

RETAILPULSE 360 transforms retail transaction data into business intelligence
by combining data cleaning, feature engineering, KPI analysis, exploratory
data analysis, visualization and business interpretation.

The analysis shows that revenue and profitability are not always aligned.
Regional performance, discounting, product-level profitability and customer
value require attention alongside headline sales.

The final output gives management a structured view of what happened, where
profitability is being created or lost, which areas require investigation,
and what actions can be considered based on evidence.

## 24. Optional Advanced Analysis — RFM Customer Segmentation

In [ ]:
# Optional RFM analysis

analysis_date = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = (
    df.groupby("Customer ID")
      .agg(
          Recency=("Order Date", lambda x: (analysis_date - x.max()).days),
          Frequency=("Order ID", "nunique"),
          Monetary=("Sales", "sum")
      )
)

rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    4,
    labels=[4,3,2,1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    4,
    labels=[1,2,3,4]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    4,
    labels=[1,2,3,4]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

display(rfm.head(10))

## Final submission checklist

- [x] Business problem
- [x] Objectives
- [x] Dataset understanding
- [x] Data inspection
- [x] Data quality checks
- [x] Data cleaning
- [x] Feature engineering
- [x] KPI analysis
- [x] Sales analysis
- [x] Profitability analysis
- [x] Product analysis
- [x] Customer analysis
- [x] Segment analysis
- [x] Regional analysis
- [x] Time-series analysis
- [x] Discount analysis
- [x] Shipping analysis
- [x] Business questions
- [x] Top insights
- [x] Recommendations
- [x] Limitations
- [x] Conclusion
- [x] Optional RFM extension